# Ablation Condition 3: Single Agent, No RAG/KG

A single LLM call per query with no retrieval-augmented generation or knowledge graph access.
The model must produce all three outputs (properties/constraints, candidate material, manufacturability) in one response,
relying purely on its parametric knowledge.

## Setup

In [1]:
import sys
import os
import json
import time
from pathlib import Path
from datetime import datetime

current_dir = Path().resolve()
if (current_dir / "src").exists() and (current_dir / "config").exists():
    project_root = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "config").exists():
    project_root = current_dir.parent
else:
    project_root = Path(os.environ.get("SYS3DEV_ROOT", current_dir.parent))
sys.path.insert(0, str(project_root))
print(f"project_root: {project_root}")

from src.config import load_config, load_prompts
from src.utils import (
    llm,
    load_ablation_queries,
    extract_json_from_response,
    build_ablation_evaluation,
    save_ablation_result,
)

config = load_config()
prompts = load_prompts()
print("Configuration and prompts loaded")

project_root: /orcd/pool/004/tphage/SG_march/MARS_ablation2
Configuration and prompts loaded


## Initialize LLM

In [2]:
llm_config = {
    "api_key": config["llm"]["api_key"],
    "base_url": config["llm"]["base_url"],
    "model": config["llm"]["model_name"],
    "max_tokens": config["llm"]["max_tokens"],
}
llm_instance = llm(llm_config)
generate = llm_instance.generate_cli
temperature = config["llm"].get("temperature", 0)
print(f"LLM initialized: {config['llm']['model_name']}")

LLM initialized: gpt-oss-20b


## Load Queries and Prompts

In [3]:
queries = load_ablation_queries()
ablation_prompts = prompts["ablation"]

print(f"Loaded {len(queries)} benchmark queries:")
for q in queries:
    print(f"  - {q['name']}: {q['sentence'][:80]}...")

Loaded 6 benchmark queries:
  - Query1: Find a substitute material for PFAS material THV (elastomeric terpolymer of tetr...
  - Query2: Identify PFAS free thermoplastic material for heat shrink tubing with a 2:1 shri...
  - Query3: Identify non fluoropolymer thermoplastics that can withstand severe thermal shoc...
  - Query4: What non-PFAS additives or formulation strategies can be used in thermoplastics ...
  - Query5: Beyond PEEK, which thermoplastic materials can:
- Withstand continuous or interm...
  - PTFE_Seals: Find a substitute material for PTFE that can be used in industrial seals applica...


## Define Single-Agent Pipeline

In [4]:
def run_1agent_no_rag_ablation(query, generate_fn, ablation_prompts, temperature=0):
    """Run the single-agent no-RAG/KG ablation for a single query."""
    raw_responses = {}
    start_time = time.time()

    print("  Running single-agent LLM call (no context)...")
    system_prompt = ablation_prompts["single_agent_no_context"]
    user_prompt = ablation_prompts["single_agent_no_context_user_prompt"].format(
        sentence=query["sentence"],
        material_X=query["material_X"],
        application_Y=query["application_Y"],
    )
    response_raw = generate_fn(system_prompt=system_prompt, prompt=user_prompt, temperature=temperature)
    raw_responses["single_agent"] = response_raw

    parsed = extract_json_from_response(response_raw)
    if parsed is None:
        print("    WARNING: Failed to parse response as JSON")
        parsed = {}

    # Extract fields
    rmp = parsed.get("required_material_properties", {})
    properties = rmp.get("properties", [])
    constraints = rmp.get("constraints", [])

    cs = parsed.get("candidate_selection", {})
    candidate = cs.get("final_candidate", cs) if cs else None

    manufacturing = parsed.get("manufacturing_process", {
        "status": "unknown",
        "process_recipe": None,
        "blocking_constraints": [],
        "feedback_to_system2": "",
    })

    print(f"    Properties: {len(properties)}, Constraints: {len(constraints)}")
    if candidate:
        print(f"    Candidate: {candidate.get('material_name', 'N/A')}")
    print(f"    Mfg status: {manufacturing.get('status', 'unknown')}")

    duration = time.time() - start_time

    return build_ablation_evaluation(
        query=query,
        properties=properties,
        constraints=constraints,
        candidate=candidate,
        manufacturing=manufacturing,
        condition_name="1agent_no_rag",
        run_id=datetime.now().strftime("%Y%m%d%H"),
        duration_seconds=duration,
        raw_responses=raw_responses,
    )

## Run All Queries

In [5]:
results = {}
output_base = os.path.join(project_root, "pipeline_logs")

for i, query in enumerate(queries, 1):
    name = query["name"]
    print(f"\n{'='*70}")
    print(f"[{i}/{len(queries)}] Running 1-Agent (no RAG/KG) ablation for: {name}")
    print(f"{'='*70}")
    print(f"  Query: {query['sentence'][:100]}...")

    result = run_1agent_no_rag_ablation(query, generate, ablation_prompts, temperature=temperature)
    results[name] = result

    output_dir = os.path.join(output_base + f"_{name}")
    path = save_ablation_result(result, output_dir, "1agent_no_rag", result["metadata"]["pipeline_run_id"])
    print(f"  Saved to: {path}")
    print(f"  Duration: {result['metadata']['duration_seconds']:.1f}s")

print(f"\n{'='*70}")
print("All 1-Agent (no RAG/KG) ablation runs complete.")
print(f"{'='*70}")


[1/6] Running 1-Agent (no RAG/KG) ablation for: Query1
  Query: Find a substitute material for PFAS material THV (elastomeric terpolymer of tetrafluoroethylene, hex...
  Running single-agent LLM call (no context)...
    Properties: 4, Constraints: 3
    Candidate: Perfluoroalkoxy Alkane (PFA) – Grade PFA-1000
    Mfg status: manufacturable
  Saved to: /orcd/pool/004/tphage/SG_march/MARS_ablation2/pipeline_logs_Query1/ablation_1agent_no_rag_2026032416.json
  Duration: 36.3s

[2/6] Running 1-Agent (no RAG/KG) ablation for: Query2
  Query: Identify PFAS free thermoplastic material for heat shrink tubing with a 2:1 shrink ratio that could ...
  Running single-agent LLM call (no context)...
    Properties: 8, Constraints: 5
    Candidate: Polyether Ether Ketone (PEEK) – Grade PEEK‑HT (e.g., 3M™ PEEK‑HT)
    Mfg status: manufacturable
  Saved to: /orcd/pool/004/tphage/SG_march/MARS_ablation2/pipeline_logs_Query2/ablation_1agent_no_rag_2026032416.json
  Duration: 41.9s

[3/6] Running 1-Agent

## Summary

In [6]:
print(f"{'Query':<20} {'Candidate':<50} {'Status':<15} {'Time (s)':<10}")
print("-" * 95)
for name, result in results.items():
    cand = result["candidate_selection"]["final_candidate"]
    cand_name = (cand.get("material_name", "N/A") if cand else "N/A")[:48]
    status = result["manufacturing_process"]["status"]
    dur = result["metadata"]["duration_seconds"]
    print(f"{name:<20} {cand_name:<50} {status:<15} {dur:<10.1f}")

Query                Candidate                                          Status          Time (s)  
-----------------------------------------------------------------------------------------------
Query1               Perfluoroalkoxy Alkane (PFA) – Grade PFA-1000      manufacturable  36.3      
Query2               Polyether Ether Ketone (PEEK) – Grade PEEK‑HT (e   manufacturable  41.9      
Query3               Polyether Ether Ketone (PEEK)                      manufacturable  19.8      
Query4               Polydimethylsiloxane (PDMS) – linear chain, Mn ≈   manufacturable  29.6      
Query5               Ultem 5000 (Polyetherimide, PEI)                   manufacturable  28.3      
PTFE_Seals           Viton® A (FKM, fluorocarbon elastomer)             manufacturable  16.3      
